In [1]:
import os
import numpy as np
import pandas as pd
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, log_loss, brier_score_loss
from sklearn.model_selection import ShuffleSplit, GridSearchCV

pd.set_option("display.max_rows", None, "display.max_columns", None)

RANDOM_STATE = 42

## Getting the data ready

In [2]:
def get_datasets(span = 5):
    path = os.path.abspath(f'../../data/dataset/men/{span}span_training_set.csv')
    training_df = pd.read_csv(path)

    path = os.path.abspath(f'../../data/dataset/men/{span}span_testing_set.csv')
    testing_df = pd.read_csv(path)

    train_true, test_true = training_df.pop('Win'), testing_df.pop('Win')

    print(f'{len(training_df)} train examples')
    print(f'{len(testing_df)} test examples')

    return training_df, testing_df, train_true, test_true

## Training the model

In [3]:
def train_model(estimator, param_grid, training_df, train_true, n_splits=3):
    # Wrap estimator in a Pipeline with StandardScaler
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', estimator)
    ])

    # Prefix param_grid keys with 'model__' for the pipeline
    pipe_param_grid = {f'model__{k}': v for k, v in param_grid.items()}

    # Use log_loss scoring to optimize probability quality (what the ensemble actually uses)
    grid_search = GridSearchCV(
        pipe, pipe_param_grid,
        cv=ShuffleSplit(n_splits, random_state=RANDOM_STATE),
        scoring='neg_log_loss',
        verbose=5,
        n_jobs=-1
    )

    grid_search.fit(training_df, train_true)

    best_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"Best hyperparameters: {best_params}")
    print(f"Best CV log_loss: {-grid_search.best_score_:.4f}")

    return grid_search.best_estimator_

## Testing the model

In [4]:
def test_model(clf, testing_df, test_true):
    y_pred = clf.predict(testing_df)
    y_proba = clf.predict_proba(testing_df)[:, 1]

    accuracy = accuracy_score(test_true, y_pred)
    logloss = log_loss(test_true, y_proba)
    brier = brier_score_loss(test_true, y_proba)

    print(f"Accuracy: {(accuracy*100):.2f}%")
    print(f"Log Loss: {logloss:.4f}")
    print(f"Brier Score: {brier:.4f}")

    print("\nClassification Report:")
    print(classification_report(test_true, y_pred))

    print("Confusion Matrix:")
    print(confusion_matrix(test_true, y_pred))

## Saving the models

In [5]:
def save_model(model, filename):
    path = os.path.abspath(f'../model/mens/{filename}')
    pickle.dump(model, open(path, 'wb'))

## Constant Variables

In [6]:
SPANS = [3, 5, 7]

In [7]:
def train_test_save(estimator, param_grid, filename, spans=[3, 5, 7], n_splits=3):
    for span in spans:
        print(f'\n{"="*50}')
        print(f'Span: {span}')
        print(f'{"="*50}')
        
        training_df, testing_df, train_true, test_true = get_datasets(span)
        clf = train_model(estimator, param_grid, training_df, train_true, n_splits)
        test_model(clf, testing_df, test_true)
        save_model(clf, f'{span}span_{filename}')

# Logistic Regression

In [8]:
# Define the Logistic Regression model
logreg_model = LogisticRegression(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['saga'],
    'max_iter': [10000]
}

filename = 'logistic_regression_model.pkl'

train_test_save(logreg_model, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best hyperparameters: {'C': 0.1, 'max_iter': 10000, 'penalty': 'l1', 'solver': 'saga'}
Best CV log_loss: 0.5715
Accuracy: 69.62%
Log Loss: 0.5684
Brier Score: 0.1942

Classification Report:
              precision    recall  f1-score   support

           0       0.65      0.52      0.58      3216
           1       0.72      0.81      0.76      4750

    accuracy                           0.70      7966
   macro avg       0.69      0.67      0.67      7966
weighted avg       0.69      0.70      0.69      7966

Confusion Matrix:
[[1687 1529]
 [ 891 3859]]

Span: 5
16747 train examples
7178 test examples
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best hyperparameters: {'C': 0.1, 'max_iter': 10000, 'penalty': 'l1', 'solver': 'saga'}
Best CV log_loss: 0.5626
Accuracy: 69.94%
Log Loss: 0.5663
Brier Score: 0.1931

Classification Report:
              precision    r

# Support Vector Machine

In [10]:
# Define the SVM hyperparameter grid
# Dropped C=10: linear+C=10 takes ~15hrs/fold and C=0.1 won span 3 anyway
param_grid = {
    'C': [0.1, 1],
    'kernel': ['rbf', 'linear'],
    'gamma': ['scale', 'auto']
}

filename = 'support_vector_machine_model.pkl'

# Span 3 already completed — only run spans 5 and 7
for span in [5, 7]:
    print(f'\n{"="*50}')
    print(f'Span: {span}')
    print(f'{"="*50}')

    training_df, testing_df, train_true, test_true = get_datasets(span)

    # Phase 1: Fast grid search (no Platt scaling)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(random_state=RANDOM_STATE))  # probability=False (default)
    ])
    pipe_param_grid = {f'model__{k}': v for k, v in param_grid.items()}

    grid_search = GridSearchCV(
        pipe, pipe_param_grid,
        cv=ShuffleSplit(3, random_state=RANDOM_STATE),
        scoring='accuracy',  # can't use log_loss without predict_proba
        verbose=5,
        n_jobs=-1
    )
    grid_search.fit(training_df, train_true)

    best_params = {k.replace('model__', ''): v for k, v in grid_search.best_params_.items()}
    print(f"Best hyperparameters: {best_params}")
    print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

    # Phase 2: Retrain once with probability=True using best params
    print("Retraining with probability=True for predict_proba support...")
    best_params['probability'] = True
    best_params['random_state'] = RANDOM_STATE
    final_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(**best_params))
    ])
    final_pipe.fit(training_df, train_true)

    test_model(final_pipe, testing_df, test_true)
    save_model(final_pipe, f'{span}span_{filename}')

# Re-evaluate span 3 from saved model
print(f'\n{"="*50}')
print(f'Span: 3 (loaded from saved model)')
print(f'{"="*50}')
training_df_3, testing_df_3, train_true_3, test_true_3 = get_datasets(3)
svm_3 = pickle.load(open(os.path.abspath(f'../model/mens/3span_{filename}'), 'rb'))
test_model(svm_3, testing_df_3, test_true_3)


Span: 5
16747 train examples
7178 test examples
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV 3/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.709 total time= 7.2min
[CV 1/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.681 total time= 7.3min
[CV 3/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.709 total time= 7.4min
[CV 2/3] END model__C=0.1, model__gamma=scale, model__kernel=rbf;, score=0.679 total time= 7.4min
[CV 2/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.679 total time= 7.4min
[CV 1/3] END model__C=0.1, model__gamma=auto, model__kernel=rbf;, score=0.681 total time= 7.5min
[CV 2/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.683 total time= 7.5min
[CV 1/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.685 total time= 7.6min
[CV 3/3] END model__C=1, model__gamma=scale, model__kernel=rbf;, score=0.719 total time= 7.9min
[CV 2/3] END model

# K-Nearest Neighbors (KNN)

In [11]:
# Define the KNN model
knn = KNeighborsClassifier()

# Define the hyperparameter grid
param_grid = {
    'n_neighbors': [3, 5, 7, 11, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
    'leaf_size': [20, 30, 50]
}

filename = 'knn_model.pkl'

train_test_save(knn, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 60 candidates, totalling 180 fits
[CV 3/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-4.551 total time=   2.5s
[CV 2/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-4.296 total time=   2.6s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=distance;, score=-4.775 total time=   3.0s
[CV 2/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-4.295 total time=   3.3s
[CV 3/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-4.550 total time=   3.3s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=3, model__p=2, model__weights=uniform;, score=-4.775 total time=   3.3s
[CV 1/3] END model__leaf_size=20, model__n_neighbors=5, model__p=2, model__weights=uniform;, score=-1.935 total time=   1.7s
[CV 1/3] EN

# Random Forests

In [12]:
# Define the Random Forest model
rfc = RandomForestClassifier(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'n_estimators': [200, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [8, 12, 20, None],
    'criterion': ['gini', 'entropy']
}

filename = 'random_forest.pkl'

train_test_save(rfc, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 32 candidates, totalling 96 fits
[CV 1/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.601 total time=  17.1s
[CV 2/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.606 total time=  17.2s
[CV 3/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=200;, score=-0.602 total time=  22.8s
[CV 3/3] END model__criterion=gini, model__max_depth=8, model__max_features=sqrt, model__n_estimators=200;, score=-0.594 total time=  35.2s
[CV 2/3] END model__criterion=gini, model__max_depth=8, model__max_features=sqrt, model__n_estimators=200;, score=-0.599 total time=  35.5s
[CV 1/3] END model__criterion=gini, model__max_depth=8, model__max_features=log2, model__n_estimators=500;, score=-0.601 total time=  41.9s
[CV 3/3] END model__criterion=gini, model__max_dep

# Gradient Boosting

In [13]:
# Define the Gradient Boosting model
gbc = GradientBoostingClassifier(random_state=RANDOM_STATE)

# Define the hyperparameter grid
param_grid = {
    'loss': ['log_loss', 'exponential'],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'max_features': ['log2', 'sqrt'],
    'n_estimators': [200, 500]
}

filename = 'gradient_boosting.pkl'

train_test_save(gbc, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 2/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.621 total time=  10.1s
[CV 1/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.618 total time=  10.3s
[CV 3/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=3, model__max_features=log2, model__n_estimators=200;, score=-0.617 total time=  11.0s
[CV 3/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=5, model__max_features=log2, model__n_estimators=200;, score=-0.603 total time=  16.5s
[CV 1/3] END model__learning_rate=0.01, model__loss=log_loss, model__max_depth=5, model__max_features=log2, model__n_estimators=200;, score=-0.603 total time=  16.8s
[CV 2/3] END model__learning_rate=0.01, model__loss=log_los

# Multilayer Perceptron

In [14]:
# Define the MLP model
mlp_model = MLPClassifier(random_state=RANDOM_STATE, early_stopping=True, max_iter=500)

# Define the hyperparameter grid
param_grid = {
    'hidden_layer_sizes': [(128, 64), (256, 128), (354, 177), (256,)],
    'activation': ['relu', 'tanh'],
    'solver': ['adam'],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate': ['adaptive']
}

filename = 'multilayer_perceptron.pkl'

train_test_save(mlp_model, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV 3/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.591 total time=  18.8s
[CV 1/3] END model__activation=relu, model__alpha=0.001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.584 total time=  19.1s
[CV 3/3] END model__activation=relu, model__alpha=0.001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.590 total time=  19.5s
[CV 1/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.586 total time=  19.6s
[CV 2/3] END model__activation=relu, model__alpha=0.0001, model__hidden_layer_sizes=(128, 64), model__learning_rate=adaptive, model__solver=adam;, score=-0.596 total time=  21.

# XGBoost

In [8]:
# Define the XGBoost model
xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    eval_metric='logloss'
)

# Define the hyperparameter grid (trimmed: 72 candidates)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'n_estimators': [200, 500],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0],
    'reg_lambda': [1, 5]
}

filename = 'xgboost.pkl'

# Span 3 already completed — only run spans 5 and 7
train_test_save(xgb_model, param_grid, filename, spans=[5, 7])

# Re-evaluate span 3 from saved model
print(f'\n{"="*50}')
print(f'Span: 3 (loaded from saved model)')
print(f'{"="*50}')
training_df_3, testing_df_3, train_true_3, test_true_3 = get_datasets(3)
xgb_3 = pickle.load(open(os.path.abspath(f'../model/mens/3span_{filename}'), 'rb'))
test_model(xgb_3, testing_df_3, test_true_3)


Span: 5
16747 train examples
7178 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.595 total time=  27.2s
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.595 total time=  27.7s
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.596 total time=  28.8s
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.594 total time=  31.6s
[CV 2/3] END model__colsample_bytree=0.8, model__

# LightGBM

In [9]:
# Define the LightGBM model
lgbm_model = LGBMClassifier(
    random_state=RANDOM_STATE,
    verbose=-1
)

# Define the hyperparameter grid (trimmed: 72 candidates)
param_grid = {
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 8],
    'n_estimators': [200, 500],
    'num_leaves': [31],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0],
    'reg_alpha': [0],
    'reg_lambda': [1, 5]
}

filename = 'lightgbm.pkl'

train_test_save(lgbm_model, param_grid, filename)


Span: 3
18585 train examples
7966 test examples
Fitting 3 folds for each of 72 candidates, totalling 216 fits
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.597 total time= 2.2min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.8;, score=-0.597 total time= 2.2min
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.8;, score=-0.596 total time= 2.3min
[CV 3/3] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=3, model__n_estimators=200, model__num_leaves=31, model__reg_alpha=0, model__reg_lambda=1, model__subsample

# Feature Importance Analysis

In [15]:
# Extract feature importances from tree-based models (span=5 as representative)
span = 5
training_df, testing_df, train_true, test_true = get_datasets(span)
feature_names = training_df.columns.tolist()

# Load the saved Random Forest and Gradient Boosting models for this span
rf_model = pickle.load(open(os.path.abspath(f'../model/mens/{span}span_random_forest.pkl'), 'rb'))
gb_model = pickle.load(open(os.path.abspath(f'../model/mens/{span}span_gradient_boosting.pkl'), 'rb'))

# Feature importance from Random Forest (pipeline: access the model step)
rf_importances = rf_model.named_steps['model'].feature_importances_
rf_importance_df = pd.DataFrame({'feature': feature_names, 'importance': rf_importances}).sort_values('importance', ascending=False)

# Feature importance from Gradient Boosting
gb_importances = gb_model.named_steps['model'].feature_importances_
gb_importance_df = pd.DataFrame({'feature': feature_names, 'importance': gb_importances}).sort_values('importance', ascending=False)

print("=== Random Forest: Top 30 Features ===")
print(rf_importance_df.head(30).to_string(index=False))

print("\n=== Gradient Boosting: Top 30 Features ===")
print(gb_importance_df.head(30).to_string(index=False))

# How much do defensive features contribute?
def_features = [f for f in feature_names if 'def_' in f]
rf_def_total = rf_importance_df[rf_importance_df['feature'].isin(def_features)]['importance'].sum()
gb_def_total = gb_importance_df[gb_importance_df['feature'].isin(def_features)]['importance'].sum()
print(f"\n=== Defensive Feature Contribution ===")
print(f"Random Forest:     {rf_def_total:.4f} ({rf_def_total*100:.1f}% of total)")
print(f"Gradient Boosting: {gb_def_total:.4f} ({gb_def_total*100:.1f}% of total)")

16747 train examples
7178 test examples
=== Random Forest: Top 30 Features ===
         feature  importance
        ORtg_CMA    0.009537
        DRtg_CMA    0.009399
    opp_ORtg_CMA    0.009121
    opp_DRtg_CMA    0.007487
    opp_ORtg_SMA    0.006895
        TRB%_CMA    0.006217
          FG_CMA    0.005773
    opp_ORtg_EMA    0.005694
    opp_TRB%_CMA    0.005548
     opp_TS%_CMA    0.005439
        ORtg_EMA    0.005002
    def_eFG%_CMA    0.004958
     opp_FG%_CMA    0.004918
    opp_DRtg_EMA    0.004897
 opp_def_FG%_CMA    0.004894
        DRtg_EMA    0.004853
     def_FG%_CMA    0.004714
     def_2P%_CMA    0.004650
         AST_CMA    0.004606
    opp_eFG%_CMA    0.004567
         TS%_CMA    0.004472
        ORtg_SMA    0.004452
    opp_DRtg_SMA    0.004410
 opp_def_2P%_CMA    0.004409
        eFG%_CMA    0.004389
         2P%_CMA    0.004345
         FG%_CMA    0.004338
      opp_FG_CMA    0.004314
        DRtg_SMA    0.004238
opp_def_eFG%_CMA    0.004231

=== Gradient Boosting

In [10]:
# Extract all model metrics for research paper
import json

model_files = {
    'Logistic Regression': 'logistic_regression_model.pkl',
    'SVM': 'support_vector_machine_model.pkl',
    'KNN': 'knn_model.pkl',
    'Random Forest': 'random_forest.pkl',
    'Gradient Boosting': 'gradient_boosting.pkl',
    'MLP': 'multilayer_perceptron.pkl',
    'XGBoost': 'xgboost.pkl',
    'LightGBM': 'lightgbm.pkl'
}

mens_results = {}
for span in [3, 5, 7]:
    training_df, testing_df, train_true, test_true = get_datasets(span)
    mens_results[span] = {}
    for name, fname in model_files.items():
        model = pickle.load(open(os.path.abspath(f'../model/mens/{span}span_{fname}'), 'rb'))
        y_pred = model.predict(testing_df)
        y_proba = model.predict_proba(testing_df)[:, 1]
        acc = accuracy_score(test_true, y_pred)
        ll = log_loss(test_true, y_proba)
        bs = brier_score_loss(test_true, y_proba)
        mens_results[span][name] = {'accuracy': round(acc*100, 2), 'log_loss': round(ll, 4), 'brier_score': round(bs, 4)}

# Print summary table
print("=" * 80)
print("MEN'S BASKETBALL - MODEL RESULTS SUMMARY")
print("=" * 80)
for span in [3, 5, 7]:
    print(f"\n--- Span {span} ---")
    print(f"{'Model':<25} {'Accuracy':>10} {'Log Loss':>10} {'Brier':>10}")
    print("-" * 55)
    for name in model_files:
        r = mens_results[span][name]
        print(f"{name:<25} {r['accuracy']:>9.2f}% {r['log_loss']:>10.4f} {r['brier_score']:>10.4f}")

# Average across spans
print(f"\n--- Average Across Spans ---")
print(f"{'Model':<25} {'Accuracy':>10} {'Log Loss':>10} {'Brier':>10}")
print("-" * 55)
for name in model_files:
    avg_acc = np.mean([mens_results[s][name]['accuracy'] for s in [3,5,7]])
    avg_ll = np.mean([mens_results[s][name]['log_loss'] for s in [3,5,7]])
    avg_bs = np.mean([mens_results[s][name]['brier_score'] for s in [3,5,7]])
    print(f"{name:<25} {avg_acc:>9.2f}% {avg_ll:>10.4f} {avg_bs:>10.4f}")

# Dump as JSON for easy copy
print("\n\nJSON_DATA_START")
print(json.dumps(mens_results, indent=2))
print("JSON_DATA_END")

18585 train examples
7966 test examples
16747 train examples
7178 test examples
15146 train examples
6492 test examples
MEN'S BASKETBALL - MODEL RESULTS SUMMARY

--- Span 3 ---
Model                       Accuracy   Log Loss      Brier
-------------------------------------------------------
Logistic Regression           69.62%     0.5684     0.1942
SVM                           69.32%     0.5743     0.1965
KNN                           63.81%     0.6439     0.2196
Random Forest                 68.83%     0.5872     0.2014
Gradient Boosting             69.02%     0.5735     0.1963
MLP                           68.96%     0.5780     0.1980
XGBoost                       69.66%     0.5708     0.1950
LightGBM                      69.65%     0.5718     0.1952

--- Span 5 ---
Model                       Accuracy   Log Loss      Brier
-------------------------------------------------------
Logistic Regression           69.94%     0.5663     0.1931
SVM                           69.49%     0.570

In [22]:
# retrain 7 span
# train_test_save(mlp_model, param_grid, filename, n_splits=1, spans=[7])

# Server testing

In [23]:
import joblib
import re

mens_model_dir, womens_model_dir = os.path.join("../model/mens/"), os.path.join("../model/womens/")
mens_filenames, womens_filenames = [filename for filename in os.listdir(mens_model_dir)], [filename for filename in os.listdir(womens_model_dir)]
mens_models, womens_models = {filename.split('.pkl')[0]: joblib.load(f'{mens_model_dir}{filename}') for filename in mens_filenames}, {filename.split('.pkl')[0]: joblib.load(f'{womens_model_dir}{filename}') for filename in womens_filenames}
print(mens_filenames)

payload = [{
    'model': '3span_ensemble_model',
    'isWomens': False,
    'team1': [
          27.33333333333333,
          26.393939393939394,
          26.58692363323644,
          58.0,
          56.90909090909091,
          56.277424356201664,
          0.4713333333333332,
          0.4665757575757576,
          0.4721426885365508,
          10.333333333333334,
          7.090909090909091,
          10.582568880636243,
          23.666666666666668,
          20.03030303030303,
          22.62090914323926,
          0.4343333333333333,
          0.3542424242424242,
          0.4643407699386589,
          10.666666666666666,
          14.787878787878787,
          11.464159086113796,
          14.333333333333334,
          19.51515151515152,
          16.123824953334406,
          0.757,
          0.7663333333333334,
          0.7228634019719903,
          6.666666666666667,
          8.575757575757576,
          6.452909055864438,
          29.0,
          30.939393939393938,
          30.71948037599213,
          16.666666666666668,
          12.424242424242424,
          16.448066715849563,
          7.333333333333333,
          6.151515151515151,
          6.840419095475227,
          1.6666666666666667,
          1.6666666666666667,
          2.218273723265156,
          9.666666666666666,
          9.424242424242424,
          9.69409596431069,
          17.666666666666668,
          15.969696969696969,
          17.0706839885097,
          113.7,
          111.4060606060606,
          113.3470722494647,
          97.56666666666666,
          102.3939393939394,
          95.66693762349897,
          66.53333333333335,
          66.7,
          66.28398357145488,
          0.2516666666666666,
          0.3579393939393939,
          0.2917529405199457,
          0.409,
          0.3535151515151515,
          0.4035162461462896,
          0.585,
          0.5660000000000001,
          0.5887726908712647,
          51.7,
          53.23030303030303,
          53.63034644359723,
          60.03333333333333,
          46.83030303030304,
          61.26416795952245,
          11.0,
          9.190909090909091,
          10.294132525194437,
          4.966666666666668,
          4.578787878787878,
          6.817281067278236,
          0.5613333333333334,
          0.5293030303030303,
          0.5664465686886107,
          12.933333333333332,
          12.412121212121212,
          13.108024106733502,
          26.066666666666663,
          29.1,
          24.82206351116765,
          0.186,
          0.272030303030303,
          0.2067422204576433
        ],
    'team2': [
          25.666666666666668,
          28.6875,
          24.59661942813545,
          55.333333333333336,
          58.625,
          54.18251581070945,
          0.4630000000000001,
          0.4889375,
          0.4534812509915791,
          7.666666666666667,
          8.4375,
          6.992306689731777,
          16.0,
          20.5625,
          16.076059906743467,
          0.4733333333333333,
          0.406125,
          0.4282937919711694,
          15.0,
          17.84375,
          16.7495296751149,
          20.666666666666668,
          24.78125,
          23.68396619334817,
          0.7303333333333333,
          0.7280625000000001,
          0.7151780618028716,
          9.0,
          11.09375,
          8.591873774304986,
          31.33333333333333,
          37.65625,
          31.69479167787358,
          20.0,
          18.40625,
          19.331843585707247,
          5.666666666666667,
          5.75,
          6.041198720224202,
          2.6666666666666665,
          3.75,
          2.86770398914814,
          10.666666666666666,
          10.84375,
          10.759370770305395,
          17.0,
          14.28125,
          16.808736738283187,
          110.06666666666666,
          118.73125,
          108.0258174452465,
          100.63333333333333,
          99.56875,
          98.5662613492459,
          67.3,
          69.8875,
          67.62328939689323,
          0.3793333333333333,
          0.42921875,
          0.4423435214064084,
          0.2903333333333334,
          0.35075,
          0.2971660725474357,
          0.5686666666666667,
          0.5944375,
          0.5581987138292752,
          53.13333333333333,
          58.196875,
          51.86511530568824,
          78.7333333333333,
          63.640625,
          79.51162182725966,
          8.433333333333335,
          8.103125,
          8.951511404290795,
          6.666666666666665,
          9.65,
          6.993118633283302,
          0.5316666666666666,
          0.560875,
          0.5177487884066068,
          14.033333333333337,
          13.38125,
          14.102322203852236,
          33.7,
          36.621875,
          31.172773850290103,
          0.276,
          0.30946875,
          0.3132329415972344
        ],
    'isNeutral': False
}]

['3span_gradient_boosting.pkl', '3span_knn_model.pkl', '3span_logistic_regression_model.pkl', '3span_multilayer_perceptron.pkl', '3span_random_forest.pkl', '3span_support_vector_machine_model.pkl', '5span_gradient_boosting.pkl', '5span_knn_model.pkl', '5span_logistic_regression_model.pkl', '5span_multilayer_perceptron.pkl', '5span_random_forest.pkl', '5span_support_vector_machine_model.pkl', '7span_gradient_boosting.pkl', '7span_knn_model.pkl', '7span_logistic_regression_model.pkl', '7span_multilayer_perceptron.pkl', '7span_random_forest.pkl', '7span_support_vector_machine_model.pkl']


In [24]:
def run():
    def get_span_number(filename):
        match = re.search(r"(\d+)span", filename)
        if match:
            return int(match.group(1))
        else:
            return None
        
    def ensemble(input, models, span):
        counter = 0
        predict_proba = [0, 0]
        for key in models:
            if f'{span}span_' in key:
                model = models[key]
                proba = model.predict_proba(input)
                predict_proba[0], predict_proba[1] = predict_proba[0] + proba[0][0], predict_proba[1] + proba[0][1]
                counter += 1
        return {
            'predict': [round(predict_proba[1] / counter)], 
            'predict_proba': [predict_proba[0] / counter, predict_proba[1] / counter]
        }

    results = []
    for matchup in payload:
        input = np.array([matchup['team2'] + matchup['team1'] + [int(matchup['isNeutral'])]])
        models = mens_models if not matchup['isWomens'] else womens_models
        if 'ensemble' in matchup['model']:
            ensemble_results = ensemble(input, models, get_span_number(matchup['model']))
            matchup['predict'], matchup['predictProba'] = ensemble_results['predict'], ensemble_results['predict_proba']
        else:
            model = models[matchup['model']]
            predict, predict_proba = model.predict(input), model.predict_proba(input)
            matchup['predict'], matchup['predictProba'] = predict.tolist(), predict_proba.tolist()[0]
        results.append(matchup)
    return results

# run()